# AMEX Enterprise Credit Risk Platform
## Notebook 39 -- Dynamic / Behavioral Credit Scoring: Modeling
### Phase 3 . Problem Statement 6: Dynamic / Behavioral Credit Scoring

CRISP-DM stage: **Modeling**. Sprint 1, Notebook 2 of 4 for this problem. Depends on Notebook 38 (reads `dynamic_behavioral_scoring_policy.json`), Problem 1 Notebooks 01-05 (raw CSV paths, real split membership, full-history champion AUC), and Problem 4's real feature list (reused verbatim, re-aggregated per window -- see Notebook 38 Section 8).

**What this notebook does (real, computed on your machine when you run it):**
- Loads Notebook 38's real policy (`TRAILING_WINDOW_CANDIDATES`, the reused 243-feature list, the KPI targets, and the standing full-metrics-suite requirement)
- Builds a genuine trailing-window feature set for each candidate `W` -- for every customer with at least `W` statements, aggregates ONLY their most recent `W` statements using the exact `_last` / `_trend_delta` / `_trend_slope` formulas Notebook 04 established for Problem 1 (cov/var slope identity, first-to-last delta, with the same inf-cleaning and null-masking correctness fixes), restricted to a fresh within-window time index computed after filtering
- Trains Problem 1's real champion windowed-GBM architecture (`n_estimators=400, max_depth=6, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, tree_method="hist", random_state=42`) independently at each candidate `W`, on the real train/holdout customer split Notebook 02 established
- Computes and displays the **full classification metrics suite** per the user's standing directive: ROC-AUC, PR-AUC (average precision), Log Loss, and the AMEX competition metric (all threshold-free); Accuracy, Precision, Recall, F1, Specificity, Matthews Correlation Coefficient, and the full confusion matrix at BOTH the standard 0.5 threshold AND a holdout-derived F1-optimal threshold (with an explicit honesty caveat that the F1-optimal threshold is chosen on the same holdout set, not a separate calibration split)
- Renders combined, multi-window ROC and Precision-Recall curve overlays inline and saves them as PNGs for the eventual report
- Reports AUC retention against Notebook 05's real full-history champion AUC, and a direct real comparison against Problem 5's own first-K AUC at the same window length (reading Problem 5's committed `notebook_35_summary.json`, no recomputation) -- an honest "recent vs. early behavioral signal" empirical finding, reported in whichever direction the real numbers show
- Writes `dynamic_behavioral_scoring_modeling_results.json` for Notebook 40 (Validation & Deployment) to consume

**What this notebook does NOT do:** no statistical validation (bootstrap CI, calibration, PSI) and no deployable scoring service -- that's Notebook 40. No financial-impact reporting -- that's Notebook 41.

Zero-fabrication: every metric this notebook prints and charts is computed live from your real Kaggle data on this run. The trailing-window candidates, feature space, and KPI targets are Notebook 38's explicit, editable `ASSUMPTION` policy, not re-decided here.

In [ ]:
# =============================================================================
# SECTION 1: ENVIRONMENT SETUP -- LOAD CONFIG FROM PROBLEM 1 (NOTEBOOKS 01-05),
#             PROBLEM 6'S OWN NOTEBOOK 38 (POLICY), AND PROBLEM 5'S REAL RESULTS
# =============================================================================
import os
import sys
import gc
import json
import time
import warnings
from pathlib import Path
from datetime import datetime, timezone


def _section(title: str) -> None:
    bar = "=" * 78
    print(f"\n{bar}\n{title}\n{bar}")


_section("SECTION 1: Environment Setup -- Load Config From Notebooks 01-05 and 38")

PROJECT_ROOT = Path(r"C:\Users\rnand\Downloads\amex-default-prediction\AMEX_Enterprise_Credit_Risk_Platform")
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
CONFIG_PATH = ARTIFACTS_DIR / "project_config.json"
NB02_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_02_summary.json"
NB05_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_05_summary.json"
NB38_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_38_summary.json"

for _p, _fix in [
    (CONFIG_PATH, "run 01_business_understanding.ipynb first"),
    (NB02_SUMMARY_PATH, "run 02_data_engineering.ipynb first"),
    (NB05_SUMMARY_PATH, "run 05_model_development.ipynb first"),
    (NB38_SUMMARY_PATH, "run 38_dynamic_behavioral_scoring_business_understanding.ipynb first "
                         "(this notebook consumes its TRAILING_WINDOW_CANDIDATES policy, not a guess)"),
]:
    if not _p.exists():
        raise FileNotFoundError(f"{_p} not found.\nFix: {_fix}")

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    PROJECT_CONFIG = json.load(f)
with open(NB02_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB02_SUMMARY = json.load(f)
with open(NB05_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB05_SUMMARY = json.load(f)
with open(NB38_SUMMARY_PATH, "r", encoding="utf-8") as f:
    NB38_SUMMARY = json.load(f)

POLICY_PATH = Path(NB38_SUMMARY["policy_path"])
if not POLICY_PATH.exists():
    raise FileNotFoundError(
        f"{POLICY_PATH} not found (path recorded in notebook_38_summary.json).\n"
        "Fix: re-run 38_dynamic_behavioral_scoring_business_understanding.ipynb."
    )
with open(POLICY_PATH, "r", encoding="utf-8") as f:
    DBS_POLICY = json.load(f)

TRAILING_WINDOW_CANDIDATES = DBS_POLICY["trailing_window_candidates"]
TRAILING_WINDOW_COVERAGE = DBS_POLICY["trailing_window_coverage_by_w"]
DBS_KPI_TARGETS = DBS_POLICY["kpi_targets"]
DBS_FEATURE_LIST = sorted(DBS_POLICY["feature_space"]["features"])

if not TRAILING_WINDOW_CANDIDATES:
    raise RuntimeError(
        "notebook_38_summary.json / dynamic_behavioral_scoring_policy.json has an "
        "empty trailing_window_candidates list -- re-run Notebook 38."
    )

# --- Optional: Problem 5's real first-K (early-window) results, for the honest
#     "recent vs. early" comparison Notebook 38 Section 6 requires. Not a hard
#     dependency -- Problem 6 does not depend on Problem 5 per the master plan
#     -- so this is read defensively and the comparison is simply skipped
#     (with a printed note) if Problem 5 hasn't been run in this environment. ---
NB35_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_35_summary.json"
P5_EARLY_WINDOW_RESULTS = {}
if NB35_SUMMARY_PATH.exists():
    with open(NB35_SUMMARY_PATH, "r", encoding="utf-8") as f:
        _nb35_summary = json.load(f)
    P5_EARLY_WINDOW_RESULTS = {int(k): v for k, v in _nb35_summary.get("results_by_k", {}).items()}
    print(f"Loaded Problem 5's real first-K results from: {NB35_SUMMARY_PATH} "
          f"(K values: {sorted(P5_EARLY_WINDOW_RESULTS.keys())})")
else:
    print(
        "NOTE: Problem 5's notebook_35_summary.json not found -- the recent-vs-early "
        "comparison in Section 8 will be skipped (Problem 6 does not depend on "
        "Problem 5, so this is informational only, not a hard requirement)."
    )

PILLAR_DIRS = {k: Path(v) for k, v in PROJECT_CONFIG["pillar_dirs"].items()}
DETECTED_LOGICAL_CORES = PROJECT_CONFIG["hardware"]["logical_cores_detected"]
RANDOM_SEED = PROJECT_CONFIG["random_seed"]

_resource_limits = PROJECT_CONFIG.get("resource_limits", {})
WARP_THREAD_COUNT = (
    _resource_limits.get("warp_thread_count")
    or PROJECT_CONFIG.get("warp_thread_count")
    or DETECTED_LOGICAL_CORES
)
MAX_RAM_BYTES = _resource_limits.get("max_ram_bytes")

if "dynamic_behavioral_scoring_modeling" in PILLAR_DIRS:
    DBS_MODELING_DIR = PILLAR_DIRS["dynamic_behavioral_scoring_modeling"]
else:
    DBS_MODELING_DIR = (
        PROJECT_ROOT / "Phase3_Behavioral_Intelligence"
        / "Problem6_Dynamic_Behavioral_Credit_Scoring" / "modeling"
    )
    print(
        "NOTE: 'dynamic_behavioral_scoring_modeling' not found in project_config.json's "
        "pillar_dirs -- using the standard folder-convention fallback:\n"
        f"      {DBS_MODELING_DIR}"
    )
DBS_MODELING_DIR.mkdir(parents=True, exist_ok=True)
CHARTS_DIR = DBS_MODELING_DIR / "charts"
CHARTS_DIR.mkdir(parents=True, exist_ok=True)

CHAMPION_NAME = NB05_SUMMARY["champion_model"]
CHAMPION_METRICS = NB05_SUMMARY["champion_metrics"]
FULL_HISTORY_AUC = CHAMPION_METRICS.get("holdout_auc")
FULL_HISTORY_AMEX_METRIC = CHAMPION_METRICS.get("holdout_amex_metric")

print(f"Loaded config from        : {CONFIG_PATH}")
print(f"Loaded trailing-window policy: {POLICY_PATH}")
print(f"RANDOM_SEED                : {RANDOM_SEED}")
print(f"WARP_THREAD_COUNT          : {WARP_THREAD_COUNT}")
print(f"Champion architecture (Problem 1, measured) : {CHAMPION_NAME}")
print(f"Champion holdout AUC (measured, reference)  : {FULL_HISTORY_AUC}")
print(f"Champion holdout AMEX metric (measured, ref): {FULL_HISTORY_AMEX_METRIC}")
print(f"TRAILING_WINDOW_CANDIDATES (from Notebook 38): {TRAILING_WINDOW_CANDIDATES}")
for _w in TRAILING_WINDOW_CANDIDATES:
    _cov = TRAILING_WINDOW_COVERAGE[str(_w)] if str(_w) in TRAILING_WINDOW_COVERAGE else TRAILING_WINDOW_COVERAGE.get(_w)
    print(f"  W={_w:>2}: {_cov:.1f}% coverage (from Notebook 38)")
print(f"Reused feature space (from Notebook 38, Problem 4's real feature list): {len(DBS_FEATURE_LIST)} features")
print(f"Modeling artifacts will be written under: {DBS_MODELING_DIR}")
print("\n\u2705 Section 1 complete.")


# =============================================================================
# SECTION 2: WARP HARDWARE CONFIGURATION & LIBRARY IMPORTS
# =============================================================================
_section("SECTION 2: WARP Hardware Configuration & Library Imports")

os.environ["POLARS_MAX_THREADS"] = str(WARP_THREAD_COUNT)
warnings.filterwarnings("ignore", category=UserWarning)

import logging
logger = logging.getLogger("amex_platform")
logger.setLevel(logging.INFO)
if not logger.handlers:
    _handler = logging.StreamHandler(sys.stdout)
    _handler.setFormatter(logging.Formatter("%(asctime)s | %(levelname)-7s | %(message)s", "%H:%M:%S"))
    logger.addHandler(_handler)

missing = []
try:
    import polars as pl
except ImportError:
    missing.append("polars")
try:
    import numpy as np
except ImportError:
    missing.append("numpy")
try:
    import psutil
except ImportError:
    missing.append("psutil")
try:
    from sklearn.metrics import (
        roc_auc_score, average_precision_score, accuracy_score, precision_score,
        recall_score, f1_score, log_loss, matthews_corrcoef, confusion_matrix,
        roc_curve, precision_recall_curve,
    )
except ImportError:
    missing.append("scikit-learn")
try:
    from xgboost import XGBClassifier
except ImportError:
    missing.append("xgboost")
try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
except ImportError:
    missing.append("matplotlib")

if missing:
    raise ImportError(
        "Missing required package(s): " + ", ".join(missing) + "\n"
        "Fix: run this in a terminal, then re-run this cell:\n"
        f"    pip install {' '.join(missing)}\n"
        "Note: xgboost is required here (not optional) because this notebook "
        "specifically tests Problem 1's real champion architecture (xgboost) "
        "at restricted trailing windows -- see Section 1's printed champion name."
    )


def _rss_gb() -> float:
    """Current process resident memory, in GB."""
    return psutil.Process().memory_info().rss / 1e9


def amex_metric_numpy(y_true: "np.ndarray", y_pred: "np.ndarray") -> float:
    """Official American Express - Default Prediction competition metric:
    0.5 * (Normalized Weighted Gini) + 0.5 * (Top-4% Capture Rate). Byte-for-
    byte the same implementation as Notebook 05 Section 3 / Notebook 35
    Section 2 -- reused here (not re-derived) so the metric definition can
    never drift between the full-history champion and this notebook's
    trailing-window models."""
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)

    def top_four_percent_captured(yt, yp):
        order = np.argsort(-yp, kind="mergesort")
        yt_sorted = yt[order]
        weight = np.where(yt_sorted == 0, 20.0, 1.0)
        cum_weight = np.cumsum(weight)
        cutoff = 0.04 * weight.sum()
        mask = cum_weight <= cutoff
        total_pos = yt_sorted.sum()
        if total_pos == 0:
            return 0.0
        return float(yt_sorted[mask].sum() / total_pos)

    def weighted_gini(yt, yp):
        order = np.argsort(-yp, kind="mergesort")
        yt_sorted = yt[order]
        weight = np.where(yt_sorted == 0, 20.0, 1.0)
        random_cum = np.cumsum(weight / weight.sum())
        total_pos_weighted = (yt_sorted * weight).sum()
        if total_pos_weighted == 0:
            return 0.0
        cum_pos_found = np.cumsum(yt_sorted * weight)
        lorentz = cum_pos_found / total_pos_weighted
        return float(((lorentz - random_cum) * weight).sum())

    g_actual = weighted_gini(y_true, y_pred)
    g_perfect = weighted_gini(y_true, y_true)
    normalized_gini = g_actual / g_perfect if g_perfect != 0 else 0.0
    top4 = top_four_percent_captured(y_true, y_pred)
    return 0.5 * (normalized_gini + top4)


logger.info(f"Polars thread pool configured to {os.environ['POLARS_MAX_THREADS']} threads (95% cap, WARP 6.4)")
print(f"Process RSS at Section 2 start: {_rss_gb():.2f} GB")
if MAX_RAM_BYTES:
    print(f"Configured RAM ceiling (90% of detected total): {MAX_RAM_BYTES / 1e9:.1f} GB")
print("amex_metric_numpy() defined (same implementation as Notebook 05 / Notebook 35).")
print("\n\u2705 Section 2 complete.")


# =============================================================================
# SECTION 3: RESOLVE REAL DATA PATHS (RAW CSVs + NOTEBOOK 02's SPLIT FILES)
# =============================================================================
_section("SECTION 3: Resolve Real Data Paths")

_raw_candidates = []
if "raw_data_dir" in PROJECT_CONFIG:
    _raw_candidates.append(Path(PROJECT_CONFIG["raw_data_dir"]) / "train_data.csv")
if "data_root" in PROJECT_CONFIG:
    _raw_candidates.append(Path(PROJECT_CONFIG["data_root"]) / "train_data.csv")
_raw_candidates.append(PROJECT_ROOT.parent / "Raw Data From Kaggle" / "train_data.csv")

RAW_TRAIN_DATA_PATH = None
for _candidate in _raw_candidates:
    if _candidate.exists() and _candidate.stat().st_size > 1_000_000:
        RAW_TRAIN_DATA_PATH = _candidate
        break

if RAW_TRAIN_DATA_PATH is None:
    raise FileNotFoundError(
        "Could not find the raw train_data.csv. Checked:\n"
        + "\n".join(f"  - {c}" for c in _raw_candidates)
        + "\n\nFix: tell me the real path to your raw train_data.csv."
    )

RAW_TRAIN_LABELS_PATH = RAW_TRAIN_DATA_PATH.parent / "train_labels.csv"
if not RAW_TRAIN_LABELS_PATH.exists():
    raise FileNotFoundError(
        f"{RAW_TRAIN_LABELS_PATH} not found (expected alongside {RAW_TRAIN_DATA_PATH.name})."
    )

print(f"Raw train_data.csv   : {RAW_TRAIN_DATA_PATH} ({RAW_TRAIN_DATA_PATH.stat().st_size / 1e9:.2f} GB)")
print(f"Raw train_labels.csv : {RAW_TRAIN_LABELS_PATH} ({RAW_TRAIN_LABELS_PATH.stat().st_size / 1e6:.2f} MB)")


def _resolve_pillar_file(filename: str, pillar_key: str, legacy_folder_name: str,
                          stored_path_str: str = None, min_size: int = 10_000) -> Path:
    """Same 3(+1)-candidate resolver every notebook in this platform uses
    (see Notebook 35 Section 3 for the full history of why): the real known
    current nested Phase/Problem path is checked FIRST (never trust
    PILLAR_DIRS alone for a pillar that predates a folder reorg), then
    PILLAR_DIRS, then the legacy root-level path, then whatever a summary
    JSON literally recorded."""
    _candidates = [
        PROJECT_ROOT / "Phase1_Foundation" / "Problem1_Credit_Scoring_PD_Prediction"
        / legacy_folder_name / filename,
    ]
    if pillar_key in PILLAR_DIRS:
        _candidates.append(PILLAR_DIRS[pillar_key] / filename)
    _candidates.append(PROJECT_ROOT / legacy_folder_name / filename)
    if stored_path_str:
        _candidates.append(Path(stored_path_str))
    for _c in _candidates:
        if _c.exists() and _c.stat().st_size > min_size:
            return _c
    raise FileNotFoundError(
        f"Could not resolve a real, non-trivial {filename}. Checked:\n"
        + "\n".join(f"  - {c}" for c in _candidates)
        + f"\n\nnotebook_02_summary.json['output_files'] keys: "
        f"{sorted(NB02_SUMMARY.get('output_files', {}).keys())}\n"
        "Fix: run the notebook that produces this file again, or tell me the real path."
    )


TRAIN_SPLIT_PATH = _resolve_pillar_file(
    "train_split.csv", "data_engineering", "Data_Engineering",
    stored_path_str=NB02_SUMMARY.get("output_files", {}).get("train_split.csv"),
)
TEST_SPLIT_PATH = _resolve_pillar_file(
    "test_split.csv", "data_engineering", "Data_Engineering",
    stored_path_str=NB02_SUMMARY.get("output_files", {}).get("test_split.csv"),
)
print(f"train_split.csv (internal train, Notebook 02's real split) : {TRAIN_SPLIT_PATH}")
print(f"test_split.csv  (internal holdout, Notebook 02's real split): {TEST_SPLIT_PATH}")
print("\n\u2705 Section 3 complete.")


# =============================================================================
# SECTION 4: LIVE SCHEMA DETECTION -- BASE RAW COLUMNS FOR THE REUSED FEATURE SPACE
# =============================================================================
_section("SECTION 4: Live Schema Detection -- Base Raw Columns")

with open(RAW_TRAIN_DATA_PATH, "r", encoding="utf-8") as f:
    _train_header = f.readline().strip().split(",")
_header_cols = set(_train_header)

# Problem 4's real feature list is suffix-tagged (_last / _trend_delta /
# _trend_slope) over BASE raw D_* columns -- recover the real base column set
# by stripping known suffixes, so only the base columns actually needed are
# streamed from the raw CSV (not all ~177 numeric columns).
_SUFFIXES = ("_trend_slope", "_trend_delta", "_last")
BASE_FEATURE_COLUMNS = set()
for _feat in DBS_FEATURE_LIST:
    for _suf in _SUFFIXES:
        if _feat.endswith(_suf):
            BASE_FEATURE_COLUMNS.add(_feat[: -len(_suf)])
            break

_missing_base_cols = BASE_FEATURE_COLUMNS - _header_cols
if _missing_base_cols:
    raise RuntimeError(
        f"{len(_missing_base_cols)} base column(s) from the reused feature list are "
        f"not present in the real raw CSV header: {sorted(_missing_base_cols)}\n"
        "Fix: this would mean Problem 4's feature list and this raw file schema have "
        "drifted apart -- investigate before proceeding rather than silently dropping columns."
    )

BASE_FEATURE_COLUMNS = sorted(BASE_FEATURE_COLUMNS)
print(f"Real base D_* raw columns needed for the reused 243-feature space: {len(BASE_FEATURE_COLUMNS)}")
print(f"Sample: {BASE_FEATURE_COLUMNS[:5]}")
print("\n\u2705 Section 4 complete.")


# =============================================================================
# SECTION 5: TRAILING-WINDOW FEATURE ENGINEERING -- REUSABLE FUNCTION
# =============================================================================
_section("SECTION 5: Trailing-Window Feature Engineering -- Reusable Function")


def build_trailing_window_store(csv_path: Path, base_cols: list, w: int) -> "pl.DataFrame":
    """Streams csv_path and returns one aggregated row per customer_ID,
    restricted to each customer's LAST W chronologically-most-recent
    statements (by real S_2 date order -- customers with fewer than W real
    statements are excluded entirely by the caller's coverage filter, not
    silently padded or truncated here).

    Computes exactly the three suffix types Problem 4's real feature list
    uses (_last, _trend_delta, _trend_slope), via the SAME cov/var slope
    identity and first-to-last delta Notebook 04 Section 4 established for
    full history -- including the same two correctness fixes documented
    there: (1) raw "inf"/"-inf" tokens cleaned to null before any cov/var
    computation, and (2) var(t) masked to null everywhere y is null, so its
    denominator is computed over exactly the same points cov(t, y) used.
    The only methodology change from Notebook 04 is WHICH rows participate:
    filtered to the trailing W statements before aggregation, not all of a
    customer's history.
    """
    schema_overrides = {"customer_ID": pl.Utf8, "S_2": pl.Utf8}
    for c in base_cols:
        schema_overrides[c] = pl.Float32

    _inf_clean_exprs = [
        pl.when(pl.col(c).is_infinite()).then(None).otherwise(pl.col(c)).alias(c)
        for c in base_cols
    ]

    lf = (
        pl.scan_csv(str(csv_path), schema_overrides=schema_overrides)
        .with_columns(pl.col("S_2").str.to_date("%Y-%m-%d"))
        .with_columns(_inf_clean_exprs)
        .sort(["customer_ID", "S_2"])
        .with_columns(
            pl.len().over("customer_ID").alias("_n_statements"),
            pl.int_range(pl.len()).over("customer_ID").alias("_row_idx"),
        )
        .filter(pl.col("_row_idx") >= (pl.col("_n_statements") - w))
        # Fresh, WITHIN-WINDOW time index (0, 1, 2, ... over just the trailing
        # W rows) -- recomputed after the filter so the slope/delta below are
        # unambiguously "trend within this window", not a global-history index.
        .with_columns(pl.int_range(pl.len()).over("customer_ID").cast(pl.Float32).alias("_t_idx"))
    )

    agg_exprs = [pl.len().alias("_actual_window_len")]
    for c in base_cols:
        agg_exprs += [
            pl.cov(pl.col("_t_idx"), pl.col(c)).alias(f"_cov_{c}"),
            pl.when(pl.col(c).is_not_null()).then(pl.col("_t_idx")).otherwise(None)
              .var().alias(f"_var_t_{c}"),
            pl.col(c).first().alias(f"_first_{c}"),
            pl.col(c).last().alias(f"{c}_last"),
        ]

    grouped = lf.group_by("customer_ID", maintain_order=False).agg(agg_exprs)

    _trend_exprs = []
    for c in base_cols:
        _trend_exprs.append(
            pl.when((pl.col(f"_var_t_{c}").is_not_null()) & (pl.col(f"_var_t_{c}") > 0))
            .then(pl.col(f"_cov_{c}") / pl.col(f"_var_t_{c}"))
            .otherwise(None)
            .alias(f"{c}_trend_slope")
        )
        _trend_exprs.append((pl.col(f"{c}_last") - pl.col(f"_first_{c}")).alias(f"{c}_trend_delta"))

    _keep_cols = ["customer_ID", "_actual_window_len"]
    _keep_cols += [f"{c}_last" for c in base_cols]
    _keep_cols += [f"{c}_trend_slope" for c in base_cols] + [f"{c}_trend_delta" for c in base_cols]

    result = grouped.with_columns(_trend_exprs).select(_keep_cols).sort("customer_ID")
    return result.collect(engine="streaming")


print("build_trailing_window_store() defined.")
print("\n\u2705 Section 5 complete.")


# =============================================================================
# SECTION 6: LOAD LABELS & TRAIN/VALIDATION SPLIT MEMBERSHIP
# =============================================================================
_section("SECTION 6: Load Labels & Train/Validation Split Membership")

labels_df = pl.read_csv(str(RAW_TRAIN_LABELS_PATH), schema_overrides={"customer_ID": pl.Utf8, "target": pl.Int8})
print(f"Live-read {RAW_TRAIN_LABELS_PATH.name}: {labels_df.shape[0]:,} labeled customers")

# --- Reuses the EXACT same train/validation split membership Notebook 02
#     established, so every candidate W's holdout AUC is measured on the
#     IDENTICAL population Notebook 05's champion AUC was measured on. ---
train_ids_set = set(pl.read_csv(str(TRAIN_SPLIT_PATH), columns=["customer_ID"])["customer_ID"].to_list())
val_ids_set = set(pl.read_csv(str(TEST_SPLIT_PATH), columns=["customer_ID"])["customer_ID"].to_list())
print(f"Train-split customers (from Notebook 02, reused)     : {len(train_ids_set):,}")
print(f"Validation-split customers (from Notebook 02, reused): {len(val_ids_set):,}")
print("\n\u2705 Section 6 complete.")


# =============================================================================
# SECTION 7: BUILD & EVALUATE THE WINDOWED-GBM CHAMPION ARCHITECTURE AT EACH
#            CANDIDATE W -- FULL CLASSIFICATION METRICS SUITE
# =============================================================================
_section("SECTION 7: Build & Evaluate the Windowed-GBM Champion at Each Candidate W")

print(
    "SCOPE (ASSUMPTION, stated plainly, same style as Notebook 35's scope note): this "
    f"notebook evaluates only Problem 1's real champion ARCHITECTURE ({CHAMPION_NAME}, "
    "same hyperparameters Notebook 05 used) at each candidate trailing window -- it does "
    "not re-run Notebook 05's full multi-model tournament at every W, for the same reason "
    "Notebook 35 gave: that would be several times the compute to answer a question this "
    "notebook isn't asking. The question here is whether RESTRICTING TO RECENT BEHAVIOR "
    "preserves predictive power, not which algorithm is best."
)
print(
    "\nTHRESHOLD METHODOLOGY NOTE (honesty caveat, per the 2026-08-25 metrics-suite "
    "directive): the 'F1-optimal' threshold below is chosen by scanning "
    "precision_recall_curve() on this SAME holdout set, not a separate calibration split "
    "-- reported for interpretability of the confusion matrix, not as an unbiased estimate "
    "of production performance. The threshold-free metrics (ROC-AUC, PR-AUC) and the "
    "standard 0.5-threshold row remain the primary, unbiased comparisons across W."
)

DBS_MODELING_RESULTS = {}
_roc_curves = {}
_pr_curves = {}

for _w in TRAILING_WINDOW_CANDIDATES:
    _section(f"  W={_w}: Building Trailing-Window Feature Set")
    gc.collect()
    _t0 = time.time()

    _base = build_trailing_window_store(RAW_TRAIN_DATA_PATH, BASE_FEATURE_COLUMNS, _w)
    # Coverage filter: only customers with a REAL, full trailing window of W
    # statements participate (matches Notebook 38's coverage percentages --
    # no silent padding of customers with fewer than W statements).
    _base = _base.filter(pl.col("_actual_window_len") == _w)

    engineered = _base.join(labels_df, on="customer_ID", how="inner")
    _build_seconds = time.time() - _t0
    print(f"W={_w}: {engineered.shape[0]:,} customers with a full {_w}-statement trailing window "
          f"x {engineered.shape[1]} columns, built in {_build_seconds:.1f}s. "
          f"Process RSS: {_rss_gb():.2f} GB")
    del _base
    gc.collect()

    train_df = engineered.filter(pl.col("customer_ID").is_in(train_ids_set))
    holdout_df = engineered.filter(pl.col("customer_ID").is_in(val_ids_set))
    del engineered
    gc.collect()

    all_feature_cols = [c for c in DBS_FEATURE_LIST if c in train_df.columns]
    if len(all_feature_cols) != len(DBS_FEATURE_LIST):
        raise RuntimeError(
            f"W={_w}: only {len(all_feature_cols)}/{len(DBS_FEATURE_LIST)} reused features "
            "were actually built -- investigate a naming mismatch before proceeding."
        )

    # Preprocessing: identical steps to Notebook 05 / Notebook 35 (inf/nan ->
    # null, median-impute fit on TRAIN split only) -- no categorical encoding
    # needed, since Problem 4's real feature list is numeric D_* only.
    _inf_clean_exprs = [
        pl.when(pl.col(c).is_infinite() | pl.col(c).is_nan()).then(None).otherwise(pl.col(c)).cast(pl.Float32).alias(c)
        for c in all_feature_cols
    ]
    train_df = train_df.with_columns(_inf_clean_exprs)
    holdout_df = holdout_df.with_columns(_inf_clean_exprs)

    _feature_medians = train_df.select(
        [pl.col(c).median().fill_null(0.0).alias(c) for c in all_feature_cols]
    ).to_dicts()[0]
    _impute_exprs = [pl.col(c).fill_null(_feature_medians[c]) for c in all_feature_cols]
    train_df = train_df.with_columns(_impute_exprs)
    holdout_df = holdout_df.with_columns(_impute_exprs)

    X_train = train_df.select(all_feature_cols).to_numpy().astype(np.float32, copy=False)
    y_train = train_df.get_column("target").to_numpy().astype(np.int64, copy=False)
    X_holdout = holdout_df.select(all_feature_cols).to_numpy().astype(np.float32, copy=False)
    y_holdout = holdout_df.get_column("target").to_numpy().astype(np.int64, copy=False)
    del train_df, holdout_df
    gc.collect()

    print(f"W={_w}: X_train {X_train.shape}, X_holdout {X_holdout.shape}, {len(all_feature_cols)} features")

    _t0 = time.time()
    model = XGBClassifier(
        n_estimators=400, max_depth=6, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
        tree_method="hist", n_jobs=WARP_THREAD_COUNT, random_state=RANDOM_SEED,
        eval_metric="auc", verbosity=0,
    )
    model.fit(X_train, y_train)
    _train_seconds = time.time() - _t0

    proba = model.predict_proba(X_holdout)[:, 1]

    # --- Threshold-free metrics ---
    holdout_auc = float(roc_auc_score(y_holdout, proba))
    holdout_pr_auc = float(average_precision_score(y_holdout, proba))
    holdout_log_loss = float(log_loss(y_holdout, proba, labels=[0, 1]))
    holdout_amex = float(amex_metric_numpy(y_holdout, proba))

    fpr, tpr, _roc_thresholds = roc_curve(y_holdout, proba)
    pr_precision, pr_recall, _pr_thresholds = precision_recall_curve(y_holdout, proba)
    _roc_curves[_w] = {"fpr": fpr.tolist(), "tpr": tpr.tolist()}
    _pr_curves[_w] = {"precision": pr_precision.tolist(), "recall": pr_recall.tolist()}

    # --- F1-optimal threshold (see honesty caveat printed above) ---
    _f1_scores = np.where(
        (pr_precision + pr_recall) > 0,
        2 * pr_precision * pr_recall / np.where((pr_precision + pr_recall) > 0, pr_precision + pr_recall, 1.0),
        0.0,
    )
    _best_idx = int(np.argmax(_f1_scores[:-1])) if len(_f1_scores) > 1 else 0
    f1_optimal_threshold = float(_pr_thresholds[_best_idx]) if len(_pr_thresholds) > 0 else 0.5

    def _threshold_metrics(threshold: float) -> dict:
        pred = (proba >= threshold).astype(np.int64)
        tn, fp, fn, tp = confusion_matrix(y_holdout, pred, labels=[0, 1]).ravel()
        specificity = float(tn / (tn + fp)) if (tn + fp) > 0 else 0.0
        return {
            "threshold": float(threshold),
            "accuracy": float(accuracy_score(y_holdout, pred)),
            "precision": float(precision_score(y_holdout, pred, zero_division=0)),
            "recall": float(recall_score(y_holdout, pred, zero_division=0)),
            "f1": float(f1_score(y_holdout, pred, zero_division=0)),
            "specificity": specificity,
            "mcc": float(matthews_corrcoef(y_holdout, pred)),
            "confusion_matrix": {"tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)},
        }

    metrics_at_050 = _threshold_metrics(0.5)
    metrics_at_f1_optimal = _threshold_metrics(f1_optimal_threshold)

    auc_retention_pct = float(holdout_auc / FULL_HISTORY_AUC * 100.0) if FULL_HISTORY_AUC else None
    meets_kpi = (
        (holdout_auc / FULL_HISTORY_AUC) >= DBS_KPI_TARGETS["min_auc_retention_vs_full_history"]
        if FULL_HISTORY_AUC else False
    )

    DBS_MODELING_RESULTS[_w] = {
        "w": _w,
        "coverage_pct": TRAILING_WINDOW_COVERAGE.get(str(_w), TRAILING_WINDOW_COVERAGE.get(_w)),
        "train_customers": int(X_train.shape[0]),
        "holdout_customers": int(X_holdout.shape[0]),
        "feature_count": len(all_feature_cols),
        "holdout_auc": holdout_auc,
        "holdout_pr_auc": holdout_pr_auc,
        "holdout_log_loss": holdout_log_loss,
        "holdout_amex_metric": holdout_amex,
        "auc_retention_pct_of_full_history": auc_retention_pct,
        "meets_kpi_target": bool(meets_kpi),
        "metrics_at_threshold_0_50": metrics_at_050,
        "metrics_at_f1_optimal_threshold": metrics_at_f1_optimal,
        "train_seconds": round(_train_seconds, 1),
    }

    print(f"\nW={_w:>2}  Holdout AUC {holdout_auc:.4f}  (retains {auc_retention_pct:.1f}% of full-history AUC "
          f"{FULL_HISTORY_AUC:.4f})  PR-AUC {holdout_pr_auc:.4f}  Log Loss {holdout_log_loss:.4f}  "
          f"AMEX {holdout_amex:.4f}  KPI: {'MET' if meets_kpi else 'NOT MET'}  (train {_train_seconds:.1f}s)")
    print(f"{'threshold':>10} {'accuracy':>9} {'precision':>10} {'recall':>8} {'f1':>7} "
          f"{'specificity':>11} {'mcc':>7}  confusion(tn,fp,fn,tp)")
    for _label, _m in (("0.50 (standard)", metrics_at_050), (f"{f1_optimal_threshold:.3f} (F1-optimal)", metrics_at_f1_optimal)):
        _cm = _m["confusion_matrix"]
        print(f"{_label:>10} {_m['accuracy']:>9.4f} {_m['precision']:>10.4f} {_m['recall']:>8.4f} "
              f"{_m['f1']:>7.4f} {_m['specificity']:>11.4f} {_m['mcc']:>7.4f}  "
              f"({_cm['tn']:,}, {_cm['fp']:,}, {_cm['fn']:,}, {_cm['tp']:,})")

    del X_train, y_train, X_holdout, y_holdout, model, proba
    gc.collect()

print(f"\nProcess RSS after all {len(TRAILING_WINDOW_CANDIDATES)} candidate windows: {_rss_gb():.2f} GB")
print("\n\u2705 Section 7 complete.")


# =============================================================================
# SECTION 8: ROC + PRECISION-RECALL CURVES (INLINE) & RETENTION SUMMARY
# =============================================================================
_section("SECTION 8: ROC + Precision-Recall Curves (Inline) & Retention Summary")

_colors = ["#0B1F3A", "#C41E3A", "#C9A227", "#2E7D32", "#6A1B9A"]

fig, ax = plt.subplots(figsize=(7, 6), dpi=150)
for _i, _w in enumerate(TRAILING_WINDOW_CANDIDATES):
    _r = DBS_MODELING_RESULTS[_w]
    _c = _roc_curves[_w]
    ax.plot(_c["fpr"], _c["tpr"], color=_colors[_i % len(_colors)], linewidth=2,
            label=f"W={_w} (AUC={_r['holdout_auc']:.4f})")
ax.plot([0, 1], [0, 1], linestyle="--", color="#8A93A6", linewidth=1, label="Random (AUC=0.500)")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title(f"Problem 6 -- ROC Curves by Trailing Window\n(reference: Notebook 05 full-history AUC {FULL_HISTORY_AUC:.4f})")
ax.legend(loc="lower right", fontsize=9)
ax.grid(alpha=0.2)
fig.tight_layout()
ROC_CHART_PATH = CHARTS_DIR / "roc_curves_by_window.png"
fig.savefig(ROC_CHART_PATH, dpi=150)
plt.show()
plt.close(fig)
print(f"Saved: {ROC_CHART_PATH}")

fig, ax = plt.subplots(figsize=(7, 6), dpi=150)
for _i, _w in enumerate(TRAILING_WINDOW_CANDIDATES):
    _r = DBS_MODELING_RESULTS[_w]
    _c = _pr_curves[_w]
    ax.plot(_c["recall"], _c["precision"], color=_colors[_i % len(_colors)], linewidth=2,
            label=f"W={_w} (PR-AUC={_r['holdout_pr_auc']:.4f})")
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Problem 6 -- Precision-Recall Curves by Trailing Window")
ax.legend(loc="upper right", fontsize=9)
ax.grid(alpha=0.2)
fig.tight_layout()
PR_CHART_PATH = CHARTS_DIR / "pr_curves_by_window.png"
fig.savefig(PR_CHART_PATH, dpi=150)
plt.show()
plt.close(fig)
print(f"Saved: {PR_CHART_PATH}")

print(f"\n{'W':>4} {'Coverage%':>10} {'Holdout AUC':>12} {'Retention%':>11} {'PR-AUC':>8} {'KPI':>8}")
for _w in TRAILING_WINDOW_CANDIDATES:
    _r = DBS_MODELING_RESULTS[_w]
    print(f"{_w:>4} {_r['coverage_pct']:>9.1f}% {_r['holdout_auc']:>12.4f} "
          f"{_r['auc_retention_pct_of_full_history']:>10.1f}% {_r['holdout_pr_auc']:>8.4f} "
          f"{'MET' if _r['meets_kpi_target'] else 'not met':>8}")

_ws_meeting_kpi = [w for w in TRAILING_WINDOW_CANDIDATES if DBS_MODELING_RESULTS[w]["meets_kpi_target"]]
if _ws_meeting_kpi:
    print(
        f"\nCandidate trailing windows meeting the "
        f"{DBS_KPI_TARGETS['min_auc_retention_vs_full_history']:.0%} AUC-retention KPI target: "
        f"{_ws_meeting_kpi}."
    )
else:
    print(
        f"\nHONEST FINDING: none of the tested trailing windows ({TRAILING_WINDOW_CANDIDATES}) retained "
        f"{DBS_KPI_TARGETS['min_auc_retention_vs_full_history']:.0%} of the full-history champion AUC on "
        "this real run. Reported plainly, not obscured."
    )

# --- Honest recent-vs-early comparison against Problem 5's real results,
#     per Notebook 38 Section 6's secondary_comparison requirement. ---
if P5_EARLY_WINDOW_RESULTS:
    print(f"\n{'W=K':>5} {'Recent (Problem 6) AUC':>24} {'Early (Problem 5) AUC':>23} {'Difference':>11}")
    for _w in TRAILING_WINDOW_CANDIDATES:
        if _w in P5_EARLY_WINDOW_RESULTS:
            _recent_auc = DBS_MODELING_RESULTS[_w]["holdout_auc"]
            _early_auc = P5_EARLY_WINDOW_RESULTS[_w]["holdout_auc"]
            _diff = _recent_auc - _early_auc
            print(f"{_w:>5} {_recent_auc:>24.4f} {_early_auc:>23.4f} {_diff:>+11.4f}")
    print(
        "\nReading this honestly: a positive difference means this customer's MOST RECENT "
        "behavior (Problem 6) carried more real predictive signal than their EARLIEST "
        "behavior (Problem 5) at the same window length; a negative difference means the "
        "opposite. Whichever direction the real numbers above show is the finding -- not "
        "assumed in either direction before this run."
    )
else:
    print("\n(Recent-vs-early comparison skipped -- Problem 5's real results were not found in this environment.)")
print("\n\u2705 Section 8 complete.")


# =============================================================================
# SECTION 9: WRITE MODELING RESULTS ARTIFACT
# =============================================================================
_section("SECTION 9: Write Modeling Results Artifact")

DBS_MODELING_ARTIFACT = {
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "problem": "Problem 6 -- Dynamic / Behavioral Credit Scoring",
    "champion_architecture_used": CHAMPION_NAME,
    "champion_architecture_scope_note": (
        "Only the champion architecture is evaluated at each W (see Section 7's "
        "printed SCOPE note) -- this is not a re-run of Notebook 05's full "
        "multi-model tournament, and not the LSTM alternative Notebook 38 "
        "Section 9 deferred."
    ),
    "full_history_reference_auc": FULL_HISTORY_AUC,
    "full_history_reference_amex_metric": FULL_HISTORY_AMEX_METRIC,
    "results_by_w": {str(w): v for w, v in DBS_MODELING_RESULTS.items()},
    "roc_curves_by_w": {str(w): v for w, v in _roc_curves.items()},
    "pr_curves_by_w": {str(w): v for w, v in _pr_curves.items()},
    "ws_meeting_kpi_target": _ws_meeting_kpi,
    "recent_vs_early_comparison": (
        {str(w): {"recent_auc": DBS_MODELING_RESULTS[w]["holdout_auc"],
                   "early_auc": P5_EARLY_WINDOW_RESULTS[w]["holdout_auc"],
                   "difference": DBS_MODELING_RESULTS[w]["holdout_auc"] - P5_EARLY_WINDOW_RESULTS[w]["holdout_auc"]}
         for w in TRAILING_WINDOW_CANDIDATES if w in P5_EARLY_WINDOW_RESULTS}
        if P5_EARLY_WINDOW_RESULTS else None
    ),
    "roc_chart_path": str(ROC_CHART_PATH),
    "pr_chart_path": str(PR_CHART_PATH),
    "threshold_methodology_note": (
        "F1-optimal threshold chosen on the same holdout set (not a separate "
        "calibration split) -- see Section 7's printed honesty caveat."
    ),
    "kpi_targets": DBS_KPI_TARGETS,
    "random_seed": RANDOM_SEED,
}

MODELING_RESULTS_PATH = DBS_MODELING_DIR / "dynamic_behavioral_scoring_modeling_results.json"
with open(MODELING_RESULTS_PATH, "w", encoding="utf-8") as f:
    json.dump(DBS_MODELING_ARTIFACT, f, indent=2)
print(f"Wrote: {MODELING_RESULTS_PATH}")
print("\n✅ Section 9 complete.")


# =============================================================================
# SECTION 10: VERIFICATION -- INTEGRITY CHECKS ON EVERYTHING THIS NOTEBOOK WROTE
# =============================================================================
_section("SECTION 10: Verification -- Integrity Checks")


def _check(label, condition, detail=""):
    status = "PASS" if condition else "FAIL"
    print(f"  [{status}] {label}" + (f" -- {detail}" if detail and not condition else ""))
    return condition


_all_checks_passed = True
_all_checks_passed &= _check("Modeling results file was written", MODELING_RESULTS_PATH.exists())
_all_checks_passed &= _check("ROC chart file was written", ROC_CHART_PATH.exists())
_all_checks_passed &= _check("PR chart file was written", PR_CHART_PATH.exists())
_all_checks_passed &= _check(
    "A result was computed for every candidate W",
    set(DBS_MODELING_RESULTS.keys()) == set(TRAILING_WINDOW_CANDIDATES),
)
_all_checks_passed &= _check(
    "Every holdout AUC is a real value in (0.5, 1.0] (better than random, at most perfect)",
    all(0.5 < r["holdout_auc"] <= 1.0 for r in DBS_MODELING_RESULTS.values()),
)
_all_checks_passed &= _check(
    "Every holdout PR-AUC is a real value in (0.0, 1.0]",
    all(0.0 < r["holdout_pr_auc"] <= 1.0 for r in DBS_MODELING_RESULTS.values()),
)
_all_checks_passed &= _check(
    "Every MCC (both thresholds) is a real value in [-1.0, 1.0]",
    all(
        -1.0 <= r["metrics_at_threshold_0_50"]["mcc"] <= 1.0
        and -1.0 <= r["metrics_at_f1_optimal_threshold"]["mcc"] <= 1.0
        for r in DBS_MODELING_RESULTS.values()
    ),
)
_all_checks_passed &= _check(
    "Every confusion matrix (both thresholds) sums to the real holdout customer count",
    all(
        sum(r["metrics_at_threshold_0_50"]["confusion_matrix"].values()) == r["holdout_customers"]
        and sum(r["metrics_at_f1_optimal_threshold"]["confusion_matrix"].values()) == r["holdout_customers"]
        for r in DBS_MODELING_RESULTS.values()
    ),
)
_all_checks_passed &= _check(
    "F1 at the F1-optimal threshold is >= F1 at the 0.5 threshold for every W "
    "(the optimal threshold was chosen BY maximizing F1 on this same holdout set)",
    all(
        r["metrics_at_f1_optimal_threshold"]["f1"] >= r["metrics_at_threshold_0_50"]["f1"] - 1e-9
        for r in DBS_MODELING_RESULTS.values()
    ),
)
_all_checks_passed &= _check(
    "auc_retention_pct is correctly derived (holdout_auc / full_history_auc)",
    all(
        abs(r["auc_retention_pct_of_full_history"] / 100.0 - (r["holdout_auc"] / FULL_HISTORY_AUC)) < 1e-6
        for r in DBS_MODELING_RESULTS.values()
    ),
)
_all_checks_passed &= _check(
    "Reused Problem 1's real champion AUC (not fabricated)",
    FULL_HISTORY_AUC == CHAMPION_METRICS.get("holdout_auc"),
)
_all_checks_passed &= _check(
    "Every W's feature matrix used exactly the reused 243-feature space (no drift)",
    all(r["feature_count"] == len(DBS_FEATURE_LIST) for r in DBS_MODELING_RESULTS.values()),
)

if not _all_checks_passed:
    raise AssertionError("One or more verification checks failed -- see FAIL lines above.")
print("\n✅ Section 10 complete -- all checks passed.")


# =============================================================================
# SECTION 11: WRITE NOTEBOOK 39 SUMMARY ARTIFACT & COMPLETION
# =============================================================================
_section("SECTION 11: Write Notebook 39 Summary Artifact")

NB39_SUMMARY = {
    "notebook": "39_dynamic_behavioral_scoring_modeling.ipynb",
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "champion_architecture_used": CHAMPION_NAME,
    "results_by_w": {str(w): v for w, v in DBS_MODELING_RESULTS.items()},
    "ws_meeting_kpi_target": _ws_meeting_kpi,
    "modeling_results_path": str(MODELING_RESULTS_PATH),
    "roc_chart_path": str(ROC_CHART_PATH),
    "pr_chart_path": str(PR_CHART_PATH),
    "random_seed": RANDOM_SEED,
}
NB39_SUMMARY_PATH = ARTIFACTS_DIR / "notebook_39_summary.json"
with open(NB39_SUMMARY_PATH, "w", encoding="utf-8") as f:
    json.dump(NB39_SUMMARY, f, indent=2)
print(f"Wrote: {NB39_SUMMARY_PATH}")

_section("NOTEBOOK 39 COMPLETE")
for _w in TRAILING_WINDOW_CANDIDATES:
    _r = DBS_MODELING_RESULTS[_w]
    print(f"  W={_w:>2}: AUC {_r['holdout_auc']:.4f} ({_r['auc_retention_pct_of_full_history']:.1f}% retention) "
          f"PR-AUC {_r['holdout_pr_auc']:.4f} -- KPI {'MET' if _r['meets_kpi_target'] else 'not met'}")
if _ws_meeting_kpi:
    print(f"\nCandidate windows meeting the KPI target: {_ws_meeting_kpi}")
else:
    print("\nNo candidate window met the KPI target on this real run (see Section 8).")
print(
    "\nNext: 40_dynamic_behavioral_scoring_validation_deployment.ipynb -- statistical "
    "validation (bootstrap CI, calibration, PSI) and a deployable FastAPI-style scoring "
    "service for the strongest viable trailing window found above."
)
